In [112]:
import gym
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from gym import spaces

class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.initial_balance = initial_balance
        self.current_step = 0
        self.cash_balance = self.initial_balance
        self.position = 0
        self.total_reward = 0
        self.last_price = self.data[self.current_step][0]
        self.position_balance = self.position * self.last_price
        self.average_buy_price = 0

        # Action space: 0 = Hold, 1 = Buy, 2 = Sell
        self.action_space = spaces.Discrete(3)

        # Observation space: [Price, Balance %, Position %, 5-day SMA]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32)

    def reset(self):
        self.current_step = 201
        self.cash_balance = self.initial_balance
        self.position = 0
        self.total_reward = 0
        self.last_price = self.data[self.current_step][0]
        self.position_balance = self.position * self.last_price
        self.average_buy_price = 0
        return self._next_observation()

    def _next_observation(self):
        if self.current_step == 0:
            prev_day = self.data[self.current_step][0]
        else:
            prev_day = self.data[self.current_step-1][0]
        return np.array([
            self.data[self.current_step][0] / prev_day,
            self.cash_balance / (self.cash_balance + self.position_balance),
            self.position_balance / (self.cash_balance + self.position_balance),
            max(1.0, self.cash_balance / self.data[self.current_step][0]),
            max(1.0, self.position),
            self.data[self.current_step][1],
            self.data[self.current_step][2],
            self.data[self.current_step][3],
        ], dtype=np.float32)

    def step(self, action):
        current_price = self.data[self.current_step][0]
        reward = 0

        self.last_price = current_price
        
        if action == 1:  # Buy
            # print(f"self.cash_balance: {self.cash_balance} current_price: {current_price}")
            if self.cash_balance >= current_price:
                if self.position > 0:
                    self.average_buy_price = ((self.average_buy_price * (self.position)) + current_price) / (self.position +1)
                else:
                    self.average_buy_price = current_price
                self.position += 1
                self.cash_balance -= current_price
                self.position_balance = self.position * self.last_price
            else:
                reward = - (self.cash_balance + self.position_balance)
        elif action == 2:  # Sell
            if self.position > 0:
                profit = (current_price - self.average_buy_price)
                reward += profit
                self.position -= 1
                self.cash_balance += current_price
            else:
                reward = - (self.cash_balance + self.position_balance)
        # else:  # Hold
        #     # NOTHING
        
        # reward += self.position_balance - (self.average_buy_price * self.position)
        # loss = self.cash_balance * 0.04 / 365
        # reward = reward - loss
        
        
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1

        obs = self._next_observation()
        self.total_reward += reward
        return obs, reward, done, {}

# Load real Uber stock data instead of random numbers
import yfinance as yf
data = yf.download("UBER", start="2020-01-01", end="2025-01-01")

data["200"] = data["Close"] / data['Close'].rolling(window=200).mean()
data["100"] = data["Close"] / data['Close'].rolling(window=100).mean()
data["20"] = data["Close"] / data['Close'].rolling(window=20).mean()

data = data[["Close", "200", "100", "20"]].to_numpy()

env = DummyVecEnv([lambda: TradingEnv(data)])

policy_kwargs = dict(net_arch=[1024, 512, 256, 128, 64])  # Three hidden layers: 256, 256, and 128 neurons

model = PPO("MlpPolicy", env, verbose=1, gamma=0.9999, learning_rate=0.0001, ent_coef=0.001, policy_kwargs=policy_kwargs)

model.learn(total_timesteps=100000)  # Train for longer

# Test the trained model
done = False
obs = env.reset()

trading_env = env.envs[0]

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    print(f"Step: {env.envs[0].current_step}, Action: {action}, Reward: {reward}, Total Reward: {env.envs[0].total_reward}, Cash Balance: {env.envs[0].cash_balance}, Position Balance: {env.envs[0].position_balance}")



/var/folders/1n/8vt634f15059yz_p3_whxnk80000gn/T/ipykernel_1579/1769914173.py:121: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level=1).reset_index()
/opt/homebrew/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/var/folders/1n/8vt634f15059yz_p3_whxnk80000gn/T/ipykernel_1579/1769914173.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing thi

datetime64[ns]
Date
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    1194
Name: count, dtype: int64
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 4560 |
|    iterations      | 1    |
|    time_elapsed    | 0    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1916        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.111186534 |
|    clip_fraction        | 0.297       |
|    clip_range           | 0.25        |
|    entropy_loss         | -1.4        |
|    explained_variance   | -1.19       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.066      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0713   

In [73]:
print(f"\n\n\n Training Done \n\n\n")




 Training Done 





In [84]:
# Download new UBER data for prediction (example: past 30 days)
# new_data = yf.download("UBER", start="2020-01-01", end="2025-01-01")
data = yf.download("NVDA", start="2024-03-01", end="2025-02-07")
data = data.stack(level=1).reset_index()
data.reset_index()
data.columns.name = None
data = make_new_features_for(data)

data = data[["Close"] + lstm_features].to_numpy()
# Create environment for new data
new_env = DummyVecEnv([lambda: TradingEnv(data)])
new_env = VecNormalize(new_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Reset environment for predictions
obs = new_env.reset()
done = False

print("\n### PREDICTIONS ON NEW DATA ###\n")

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = new_env.step(action)
    print(f"Step: {new_env.envs[0].current_step}, Action: {action}, Reward: {reward}, Position: {new_env.envs[0].position} Last Price: {new_env.envs[0].last_price} Total Balance: {new_env.envs[0].cash_balance + new_env.envs[0].position * new_env.envs[0].last_price} Total Reward: {new_env.envs[0].total_reward}, Cash Balance: {new_env.envs[0].cash_balance}, Position Balance: {new_env.envs[0].position * new_env.envs[0].last_price}")


[*********************100%***********************]  1 of 1 completed

datetime64[ns]
Date
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    235
Name: count, dtype: int64

### PREDICTIONS ON NEW DATA ###

Step: 202, Action: [[1.]], Reward: [-1.3336064], Position: 76 Last Price: 130.38999938964844 Total Balance: 9980.18845058773 Total Reward: -0.013337249895192915, Cash Balance: 70.54849697444885, Position Balance: 9909.639953613281
Step: 203, Action: [[0.]], Reward: [-1.47547], Position: 0 Last Price: 128.91000366210938 Total Balance: 9849.193700157819 Total Reward: -0.02863802629576484, Cash Balance: 9849.193700157819, Position Balance: 0.0
Step: 204, Action: [[0.]], Reward: [0.], Position: 0 Last Price: 130.67999267578125 Total Balance: 9850.273063850987 Total Reward: -0.02863802629576484, Cash Balance: 9850.273063850987, Position Balance: 0.0
Step: 205, Action: [[0.]], Reward: [0.], Position: 0 Last Price: 134.6999969482422 Total Balance: 9851.352545830587 Total Reward: -0.02863802629576484, Cash Balance: 9851.352545830587, Position Balance: 0.0
St


/var/folders/1n/8vt634f15059yz_p3_whxnk80000gn/T/ipykernel_1579/2868570479.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level=1).reset_index()
/opt/homebrew/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/var/folders/1n/8vt634f15059yz_p3_whxnk80000gn/T/ipykernel_1579/1839566434.py:81: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this 